# RoadFlood-VLM Ablation Experiments

Inference-time ablations using the trained adapter and the same test split as `11_evaluation.ipynb`.

Configurations:

1. Full: Sentinel-2 + Sentinel-1 + transportation grounding
2. Optical only: Sentinel-2 + transportation grounding
3. SAR only: Sentinel-1 + transportation grounding
4. Vision only: Sentinel-2 + Sentinel-1 without explicit grounding facts or labels
5. Grounding only: transportation grounding without images

These experiments do not retrain the model.

In [ ]:
from __future__ import annotations
import gc, json, math, os, re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from rouge_score import rouge_scorer

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate/'data').exists() and (candidate/'notebooks').exists():
            return candidate
    raise FileNotFoundError('Run from the ResilientVLM repository or a subdirectory.')

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT/'data'/'processed'/'vlm_dataset'
SPLIT_DIR = DATASET_ROOT/'splits'
TRAINING_ROOT = PROJECT_ROOT/'outputs'/'roadflood_vlm_training'
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = PROJECT_ROOT/'outputs'/'roadflood_vlm_ablation'/f'ablation_{STAMP}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = os.environ.get('ROADFLOOD_MODEL_ID','Qwen/Qwen2.5-VL-3B-Instruct').strip()
raw_adapter = os.environ.get('ROADFLOOD_ADAPTER_PATH','').strip()
if raw_adapter:
    ADAPTER_DIR = Path(raw_adapter).expanduser().resolve()
else:
    runs = sorted(p for p in TRAINING_ROOT.glob('full_*') if (p/'final_adapter').exists())
    if not runs: raise FileNotFoundError(f'No final adapter under {TRAINING_ROOT}')
    ADAPTER_DIR = runs[-1]/'final_adapter'
if not ADAPTER_DIR.exists(): raise FileNotFoundError(ADAPTER_DIR)

TEST_CSV = SPLIT_DIR/'test.csv'
if not TEST_CSV.exists(): raise FileNotFoundError(TEST_CSV)
MAX_RECORDS = int(os.environ.get('ROADFLOOD_ABLATION_MAX_RECORDS','0'))
MAX_NEW_TOKENS = int(os.environ.get('ROADFLOOD_MAX_NEW_TOKENS','256'))
print('ADAPTER_DIR:', ADAPTER_DIR)
print('RUN_DIR:', RUN_DIR)

In [ ]:
TEXT_COLUMNS=['multimodal_prompt','instruction_text','instruction','input']
RESPONSE_COLUMNS=['response_text','response']
S2_COLUMNS=['training_s2_path','s2_png_path','training_s2_relative_path','s2_png_relative_path']
S1_COLUMNS=['training_s1_path','s1_png_path','training_s1_relative_path','s1_png_relative_path']

def first_col(df,candidates,label):
    for c in candidates:
        if c in df.columns: return c
    raise KeyError(f'No {label} column; expected one of {candidates}')

def resolve_path(value: Any) -> Path:
    raw='' if pd.isna(value) else str(value).strip()
    if not raw: return Path('')
    direct=Path(raw)
    if direct.is_absolute() and direct.exists(): return direct
    norm=raw.replace('\\','/')
    for marker in ['data/','training_images/']:
        i=norm.find(marker)
        if i>=0:
            suffix=Path(norm[i:])
            for root in [PROJECT_ROOT,DATASET_ROOT]:
                p=root/suffix
                if p.exists(): return p.resolve()
    for root in [PROJECT_ROOT,DATASET_ROOT,SPLIT_DIR]:
        p=root/direct
        if p.exists(): return p.resolve()
    return direct

test_df=pd.read_csv(TEST_CSV)
text_col=first_col(test_df,TEXT_COLUMNS,'prompt')
response_col=first_col(test_df,RESPONSE_COLUMNS,'response')
s2_col=first_col(test_df,S2_COLUMNS,'Sentinel-2')
s1_col=first_col(test_df,S1_COLUMNS,'Sentinel-1')
test_df['prompt']=test_df[text_col].fillna('').astype(str).str.strip()
test_df['reference']=test_df[response_col].fillna('').astype(str).str.strip()
test_df['s2']=test_df[s2_col].map(resolve_path)
test_df['s1']=test_df[s1_col].map(resolve_path)
valid=(test_df.prompt.ne('') & test_df.reference.ne('') & test_df.s2.map(Path.exists) & test_df.s1.map(Path.exists))
invalid_df=test_df.loc[~valid].copy()
if not invalid_df.empty: invalid_df.to_csv(RUN_DIR/'invalid_test_records.csv',index=False)
test_df=test_df.loc[valid].reset_index(drop=True)
if MAX_RECORDS>0: test_df=test_df.head(MAX_RECORDS).copy()
if test_df.empty: raise RuntimeError('No valid test records remain.')
print('Valid records:',len(test_df))

In [ ]:
use_4bit=(os.environ.get('ROADFLOOD_USE_4BIT','1')=='1' and torch.cuda.is_available())
dtype=(torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32)
quant=None
if use_4bit:
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=dtype,bnb_4bit_use_double_quant=True)
processor=AutoProcessor.from_pretrained(ADAPTER_DIR,trust_remote_code=True)
if processor.tokenizer.pad_token_id is None: processor.tokenizer.pad_token=processor.tokenizer.eos_token
kwargs={'trust_remote_code':True,'torch_dtype':dtype,'low_cpu_mem_usage':True}
if torch.cuda.is_available(): kwargs['device_map']='auto'
if quant is not None: kwargs['quantization_config']=quant
base_model=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID,**kwargs)
model=PeftModel.from_pretrained(base_model,ADAPTER_DIR)
model.eval(); model.config.use_cache=True
print('Model loaded; 4-bit:',use_4bit,'dtype:',dtype)

In [ ]:
ABLATIONS={
 'full':{'use_s2':True,'use_s1':True,'use_grounding':True},
 'optical_only':{'use_s2':True,'use_s1':False,'use_grounding':True},
 'sar_only':{'use_s2':False,'use_s1':True,'use_grounding':True},
 'vision_only':{'use_s2':True,'use_s1':True,'use_grounding':False},
 'grounding_only':{'use_s2':False,'use_s1':False,'use_grounding':True},
}
SYSTEM_PROMPTS={
 'full':'You are RoadFlood-VLM, a transportation-flood assessment assistant. Use the Sentinel-2 optical image, Sentinel-1 radar image, and supplied transportation context. Produce a concise, evidence-grounded response.',
 'optical_only':'You are RoadFlood-VLM, a transportation-flood assessment assistant. Use the Sentinel-2 optical image and supplied transportation context. Produce a concise, evidence-grounded response.',
 'sar_only':'You are RoadFlood-VLM, a transportation-flood assessment assistant. Use the Sentinel-1 radar image and supplied transportation context. Produce a concise, evidence-grounded response.',
 'vision_only':'You are RoadFlood-VLM, a transportation-flood assessment assistant. Use only the supplied Sentinel-2 optical and Sentinel-1 radar images. Do not assume numerical facts that are not visually supported.',
 'grounding_only':'You are RoadFlood-VLM, a transportation-flood assessment assistant. Use only the supplied transportation-grounding text. No imagery is available.',
}
pd.DataFrame(ABLATIONS).T

In [ ]:
GROUNDING_PATTERNS=[
 re.compile(r'^\s*Scene\s*:',re.I), re.compile(r'^\s*Physical roadway edges\s*:',re.I),
 re.compile(r'^\s*Edges with valid raster coverage\s*:',re.I), re.compile(r'^\s*Flood-exposed edges\s*:',re.I),
 re.compile(r'^\s*Flood-exposed edge share\s*:',re.I), re.compile(r'^\s*Low exposure edges\s*:',re.I),
 re.compile(r'^\s*Moderate exposure edges\s*:',re.I), re.compile(r'^\s*High exposure edges\s*:',re.I),
 re.compile(r'^\s*Scene flood burden\s*:',re.I), re.compile(r'^\s*Critical-network disruption\s*:',re.I),
 re.compile(r'^\s*Grounding reliability\s*:',re.I), re.compile(r'^\s*Transportation flood disruption',re.I),
 re.compile(r'^\s*Combined transportation',re.I),
]

def strip_grounding(prompt,row):
    text=str(prompt)
    text=re.sub(r'The first image is a Sentinel-2 true-color optical image\.\s*The second image is a Sentinel-1 radar polarization visualization\.\s*','',text,flags=re.I)
    text=re.sub(r'Use both images together with the transportation-grounding information implied by the question\.\s*','',text,flags=re.I)
    text='\n'.join(line for line in text.splitlines() if not any(p.search(line) for p in GROUNDING_PATTERNS))
    text=re.sub(r'Scene flood burden\s*:\s*(None|Low|Moderate|High)\.?','',text,flags=re.I)
    text=re.sub(r'Critical-network disruption\s*:\s*(None|Low|Moderate|High)\.?','',text,flags=re.I)
    text=re.sub(r'Grounding reliability\s*:\s*(None|Low|Moderate|High)\.?','',text,flags=re.I)
    instruction=str(row.get('instruction_text',row.get('instruction',''))).strip()
    if 'Task:' in text: text=text[text.index('Task:'):]
    elif instruction: text='Task: '+instruction
    text=re.sub(r'\n{3,}','\n\n',text).strip()
    return text+'\nUse only visible image evidence. Do not infer exact roadway counts, percentages, scene identifiers, or assigned categories unless visually supported.'

def adapt_prompt(row,experiment):
    text=str(row['prompt'])
    if experiment=='vision_only': return strip_grounding(text,row)
    if experiment=='full': return text
    if experiment=='optical_only':
        text=re.sub(r'The first image is a Sentinel-2 true-color optical image\.\s*The second image is a Sentinel-1 radar polarization visualization\.\s*','The image is a Sentinel-2 true-color optical image. ',text,flags=re.I)
        return text.replace('Use both images','Use the optical image')
    if experiment=='sar_only':
        text=re.sub(r'The first image is a Sentinel-2 true-color optical image\.\s*The second image is a Sentinel-1 radar polarization visualization\.\s*','The image is a Sentinel-1 radar polarization visualization. ',text,flags=re.I)
        return text.replace('Use both images','Use the radar image')
    if experiment=='grounding_only':
        text=re.sub(r'The first image is a Sentinel-2 true-color optical image\.\s*The second image is a Sentinel-1 radar polarization visualization\.\s*','',text,flags=re.I)
        return text.replace('Use both images together with the transportation-grounding information implied by the question.','Use the transportation-grounding information supplied in the question.').strip()
    raise ValueError(experiment)

def build_messages(row,experiment):
    cfg=ABLATIONS[experiment]; content=[]
    if cfg['use_s2']: content.append({'type':'image','image':str(row['s2'])})
    if cfg['use_s1']: content.append({'type':'image','image':str(row['s1'])})
    content.append({'type':'text','text':adapt_prompt(row,experiment)})
    return [{'role':'system','content':[{'type':'text','text':SYSTEM_PROMPTS[experiment]}]},{'role':'user','content':content}]

for exp in ABLATIONS:
    print('\n---',exp,'---')
    print(adapt_prompt(test_df.iloc[0],exp)[:900])

In [ ]:
def normalize_text(text):
    return re.sub(r'[^\w\s]','',re.sub(r'\s+',' ',str(text).lower().strip()))

def token_f1(reference,prediction):
    r=normalize_text(reference).split(); p=normalize_text(prediction).split()
    if not r and not p: return 1.0
    if not r or not p: return 0.0
    overlap=sum((Counter(r)&Counter(p)).values()); precision=overlap/len(p); recall=overlap/len(r)
    return 2*precision*recall/(precision+recall) if precision+recall else 0.0

def generate_prediction(row,experiment):
    messages=build_messages(row,experiment)
    text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    image_inputs,video_inputs=process_vision_info(messages)
    kwargs={'text':[text],'padding':True,'return_tensors':'pt'}
    if image_inputs: kwargs['images']=[image_inputs]
    if video_inputs: kwargs['videos']=[video_inputs]
    batch=processor(**kwargs)
    device=next(model.parameters()).device
    batch={k:(v.to(device) if isinstance(v,torch.Tensor) else v) for k,v in batch.items()}
    prompt_len=batch['input_ids'].shape[1]
    with torch.inference_mode():
        ids=model.generate(**batch,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,use_cache=True)
    pred=processor.batch_decode(ids[:,prompt_len:],skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()
    del batch,ids; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pred

def run_experiment(experiment,df):
    d=RUN_DIR/experiment; d.mkdir(parents=True,exist_ok=True); out=d/'predictions.csv'
    if out.exists():
        existing=pd.read_csv(out); completed=set(pd.to_numeric(existing.get('record_index',pd.Series(dtype=int)),errors='coerce').dropna().astype(int)); records=existing.to_dict('records')
        print(experiment,'resuming with',len(completed),'records')
    else: completed=set(); records=[]
    for i,row in df.iterrows():
        if int(i) in completed: continue
        pred=generate_prediction(row,experiment); ref=row['reference']
        rec={'experiment':experiment,'record_index':int(i),'instruction_id':str(row.get('instruction_id',f'record_{i}')),'scene_id':str(row.get('scene_id','')),'task_family':str(row.get('task_family','')),'prompt':row['prompt'],'ablation_prompt':adapt_prompt(row,experiment),'reference':ref,'prediction':pred,'exact_match':float(normalize_text(ref)==normalize_text(pred)),'token_f1':token_f1(ref,pred),'s2_used':ABLATIONS[experiment]['use_s2'],'s1_used':ABLATIONS[experiment]['use_s1'],'grounding_used':ABLATIONS[experiment]['use_grounding'],'s2_path':str(row['s2']),'s1_path':str(row['s1'])}
        records.append(rec); pd.DataFrame(records).sort_values('record_index').to_csv(out,index=False)
        print(f'[{experiment}] [{i+1}/{len(df)}] scene={rec["scene_id"]} token_f1={rec["token_f1"]:.4f}')
    return out

## Run the ablations

For a two-record smoke test before the full run:

```bash
export ROADFLOOD_ABLATION_MAX_RECORDS=2
```

Use `0` or unset the variable for all test records. The prediction files are written after every record, so interrupted experiments can resume.

In [ ]:
prediction_files={}
for experiment in ABLATIONS:
    print('\n'+'#'*80+'\nRUNNING '+experiment+'\n'+'#'*80)
    prediction_files[experiment]=run_experiment(experiment,test_df)
print(prediction_files)

In [ ]:
NUMBER_PATTERN=re.compile(r'(?<![\w.])-?\d+(?:,\d{3})*(?:\.\d+)?%?')
SCENE_ID_PATTERN=re.compile(r'"scene_id"\s*:\s*"([^"]+)"',re.I)
FLOOD_PATTERNS=[re.compile(r'"scene_flood_burden"\s*:\s*"(None|Low|Moderate|High)"',re.I),re.compile(r'(?:scene has a|identifies a|roadway flood burden is)\s+(None|Low|Moderate|High)\s+(?:roadway )?flood burden',re.I),re.compile(r'roadway flood burden(?: class)? is\s+(None|Low|Moderate|High)',re.I)]
DISRUPTION_PATTERNS=[re.compile(r'"critical_network_disruption"\s*:\s*"(None|Low|Moderate|High)"',re.I),re.compile(r'critical-network disruption(?: category| class)? is\s+(None|Low|Moderate|High)',re.I),re.compile(r'(None|Low|Moderate|High)\s+critical-network disruption',re.I)]

def simple_tokens(text): return re.findall(r'\b\w+(?:[.-]\w+)*\b',str(text).lower())
def normalize_number(token):
    token=token.replace(',','').strip(); pct=token.endswith('%'); token=token[:-1] if pct else token
    try:
        n=float(token); out=str(int(round(n))) if math.isclose(n,round(n)) else f'{n:.6f}'.rstrip('0').rstrip('.')
    except ValueError: out=token
    return out+'%' if pct else out
def extract_numbers(text): return [normalize_number(x) for x in NUMBER_PATTERN.findall(str(text))]
def prf(ref,pred):
    rc,pc=Counter(ref),Counter(pred); overlap=sum((rc&pc).values()); precision=overlap/sum(pc.values()) if pc else float(not rc); recall=overlap/sum(rc.values()) if rc else float(not pc); f1=2*precision*recall/(precision+recall) if precision+recall else 0.0; return precision,recall,f1
def try_json(text):
    s=str(text).strip()
    if not(s.startswith('{') and s.endswith('}')): return None
    try: x=json.loads(s)
    except json.JSONDecodeError: return None
    return x if isinstance(x,dict) else None
def extract_scene(text):
    x=try_json(text)
    if x is not None and 'scene_id' in x: return str(x['scene_id']).strip()
    m=SCENE_ID_PATTERN.search(str(text)); return m.group(1).strip() if m else None
def extract_cat(text,patterns):
    for p in patterns:
        m=p.search(str(text))
        if m: return m.group(1).title()
    return None
def safe_mean(x):
    s=pd.to_numeric(pd.Series(x),errors='coerce').dropna(); return float(s.mean()) if not s.empty else None

In [ ]:
rouge=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=True)
smoothing=SmoothingFunction().method1

def calculate_metrics(path):
    df=pd.read_csv(path); refs=df.reference.fillna('').astype(str).tolist(); preds=df.prediction.fillna('').astype(str).tolist()
    bleu=corpus_bleu([[simple_tokens(r)] for r in refs],[simple_tokens(p) for p in preds],smoothing_function=smoothing)
    rows=[]
    for ref,pred in zip(refs,preds):
        rv=rouge.score(ref,pred); rn,pn=extract_numbers(ref),extract_numbers(pred); np_,nr,nf=prf(rn,pn); rj,pj=try_json(ref),try_json(pred); rs,ps=extract_scene(ref),extract_scene(pred); rb,pb=extract_cat(ref,FLOOD_PATTERNS),extract_cat(pred,FLOOD_PATTERNS); rd,pd_=extract_cat(ref,DISRUPTION_PATTERNS),extract_cat(pred,DISRUPTION_PATTERNS)
        if rj is None: jv=jkr=jva=np.nan
        elif pj is None: jv,jkr,jva=0.0,0.0,0.0
        else:
            keys=set(rj); match=keys&set(pj); jv=1.0; jkr=len(match)/len(keys) if keys else 1.0; jva=sum(str(pj[k]).strip().lower()==str(rj[k]).strip().lower() for k in match)/len(keys) if keys else 1.0
        rows.append({'rouge1_f1':rv['rouge1'].fmeasure,'rouge2_f1':rv['rouge2'].fmeasure,'rougeL_f1':rv['rougeL'].fmeasure,'numeric_precision':np_,'numeric_recall':nr,'numeric_f1':nf,'numeric_exact_match':float(Counter(rn)==Counter(pn)),'scene_id_accuracy':float(rs is not None and ps is not None and rs.lower()==ps.lower()) if rs is not None else np.nan,'flood_burden_accuracy':float(rb is not None and pb is not None and rb==pb) if rb is not None else np.nan,'critical_disruption_accuracy':float(rd is not None and pd_ is not None and rd==pd_) if rd is not None else np.nan,'json_validity':jv,'json_key_recall':jkr,'json_value_accuracy':jva})
    extra=pd.DataFrame(rows)
    for c in extra.columns: df[c]=extra[c]
    metrics={'experiment':str(df.experiment.iloc[0]),'records':int(len(df)),'text_generation':{'bleu':float(bleu),'rouge1_f1':safe_mean(df.rouge1_f1),'rouge2_f1':safe_mean(df.rouge2_f1),'rougeL_f1':safe_mean(df.rougeL_f1),'mean_token_f1':safe_mean(df.token_f1),'exact_match':safe_mean(df.exact_match)},'grounding_and_factuality':{'numeric_precision':safe_mean(df.numeric_precision),'numeric_recall':safe_mean(df.numeric_recall),'numeric_f1':safe_mean(df.numeric_f1),'numeric_exact_match':safe_mean(df.numeric_exact_match),'scene_id_accuracy':safe_mean(df.scene_id_accuracy),'flood_burden_accuracy':safe_mean(df.flood_burden_accuracy),'critical_disruption_accuracy':safe_mean(df.critical_disruption_accuracy)},'structured_outputs':{'json_validity':safe_mean(df.json_validity),'json_key_recall':safe_mean(df.json_key_recall),'json_value_accuracy':safe_mean(df.json_value_accuracy)}}
    return metrics,df

### Optional BERTScore

BERTScore is off by default to reduce GPU memory and runtime. Enable it before execution with:

```bash
export ROADFLOOD_ABLATION_BERTSCORE=1
```

In [ ]:
RUN_BERTSCORE=os.environ.get('ROADFLOOD_ABLATION_BERTSCORE','0')=='1'
bertscorer=None
if RUN_BERTSCORE:
    from bert_score import BERTScorer
    bertscorer=BERTScorer(model_type=os.environ.get('ROADFLOOD_BERTSCORE_MODEL','microsoft/deberta-xlarge-mnli'),device='cuda' if torch.cuda.is_available() else 'cpu',batch_size=int(os.environ.get('ROADFLOOD_BERTSCORE_BATCH_SIZE','4')),rescale_with_baseline=False)
    bertscorer._tokenizer.model_max_length=512
print('BERTScore enabled:',RUN_BERTSCORE)

In [ ]:
all_metrics={}
for experiment,path in prediction_files.items():
    metrics,df=calculate_metrics(path)
    if bertscorer is not None:
        refs=df.reference.fillna('').astype(str).tolist(); preds=df.prediction.fillna('').astype(str).tolist(); bp,br,bf=bertscorer.score(preds,refs,verbose=True); df['bertscore_precision']=bp.cpu().numpy(); df['bertscore_recall']=br.cpu().numpy(); df['bertscore_f1']=bf.cpu().numpy(); metrics['text_generation'].update({'bertscore_precision':safe_mean(df.bertscore_precision),'bertscore_recall':safe_mean(df.bertscore_recall),'bertscore_f1':safe_mean(df.bertscore_f1)})
    else: metrics['text_generation'].update({'bertscore_precision':None,'bertscore_recall':None,'bertscore_f1':None})
    d=RUN_DIR/experiment; df.to_csv(d/'predictions_extended_metrics.csv',index=False); (d/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8'); all_metrics[experiment]=metrics
    print(json.dumps(metrics,indent=2))

In [ ]:
rows=[]
for exp,m in all_metrics.items():
    t=m['text_generation']; g=m['grounding_and_factuality']; s=m['structured_outputs']
    rows.append({'configuration':exp,'sentinel_2':ABLATIONS[exp]['use_s2'],'sentinel_1':ABLATIONS[exp]['use_s1'],'transportation_grounding':ABLATIONS[exp]['use_grounding'],'records':m['records'],'bleu':t['bleu'],'rouge1_f1':t['rouge1_f1'],'rouge2_f1':t['rouge2_f1'],'rougeL_f1':t['rougeL_f1'],'bertscore_f1':t['bertscore_f1'],'token_f1':t['mean_token_f1'],'numeric_f1':g['numeric_f1'],'numeric_exact_match':g['numeric_exact_match'],'scene_id_accuracy':g['scene_id_accuracy'],'flood_burden_accuracy':g['flood_burden_accuracy'],'critical_disruption_accuracy':g['critical_disruption_accuracy'],'json_validity':s['json_validity'],'json_key_recall':s['json_key_recall'],'json_value_accuracy':s['json_value_accuracy']})
summary_df=pd.DataFrame(rows)
order=list(ABLATIONS); summary_df['configuration']=pd.Categorical(summary_df.configuration,categories=order,ordered=True); summary_df=summary_df.sort_values('configuration').reset_index(drop=True); summary_df['configuration']=summary_df.configuration.astype(str)
summary_csv=RUN_DIR/'ablation_summary.csv'; summary_json=RUN_DIR/'ablation_summary.json'; summary_df.to_csv(summary_csv,index=False); summary_json.write_text(json.dumps({'model_id':MODEL_ID,'adapter_dir':str(ADAPTER_DIR),'run_dir':str(RUN_DIR),'bert_score_enabled':RUN_BERTSCORE,'experiments':all_metrics},indent=2),encoding='utf-8')
display(summary_df.round(4)); print(summary_csv); print(summary_json)

In [ ]:
baseline=summary_df.set_index('configuration').loc['full']; metrics=['bleu','rougeL_f1','bertscore_f1','token_f1','numeric_f1','json_value_accuracy']; delta=[]
for _,row in summary_df.iterrows():
    x={'configuration':row.configuration}
    for metric in metrics:
        b=baseline[metric]; v=row[metric]; x[metric+'_absolute_change']=v-b if pd.notna(v) and pd.notna(b) else np.nan; x[metric+'_percent_change']=100*(v-b)/b if pd.notna(v) and pd.notna(b) and b!=0 else np.nan
    delta.append(x)
delta_df=pd.DataFrame(delta); delta_csv=RUN_DIR/'ablation_change_from_full.csv'; delta_df.to_csv(delta_csv,index=False); display(delta_df.round(4)); print(delta_csv)

In [ ]:
import matplotlib.pyplot as plt
plot_df=summary_df.set_index('configuration')[['rougeL_f1','token_f1','numeric_f1','json_value_accuracy']]
ax=plot_df.plot(kind='bar',figsize=(12,6)); ax.set_title('RoadFlood-VLM Ablation Performance'); ax.set_xlabel('Ablation configuration'); ax.set_ylabel('Score'); ax.set_ylim(0,1); ax.tick_params(axis='x',rotation=25); ax.legend(title='Metric'); plt.tight_layout(); figure_path=RUN_DIR/'ablation_performance.png'; plt.savefig(figure_path,dpi=300,bbox_inches='tight'); plt.show(); print(figure_path)

## Interpretation guidance

- A decline from the full configuration indicates that the removed modality contributes to response quality.
- Strong language-overlap scores with weak numeric scores indicate fluent template reproduction without reliable factual grounding.
- The grounded prompts already provide flood-burden and disruption labels. Their accuracy is therefore not independent classification accuracy.
- The vision-only condition is intentionally harder because explicit labels and numerical grounding are removed.
- A later label-hidden evaluation should test category inference under otherwise comparable prompts.

In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'model_id':MODEL_ID,'adapter_dir':str(ADAPTER_DIR),'test_csv':str(TEST_CSV),'run_dir':str(RUN_DIR),'max_new_tokens':MAX_NEW_TOKENS,'use_4bit':use_4bit,'dtype':str(dtype),'records_per_experiment':int(len(test_df)),'bert_score_enabled':RUN_BERTSCORE,'prediction_files':{k:str(v) for k,v in prediction_files.items()},'summary_csv':str(summary_csv),'summary_json':str(summary_json),'change_from_full_csv':str(delta_csv),'figure':str(figure_path)}
manifest_path=RUN_DIR/'run_manifest.json'; manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8'); print(json.dumps(manifest,indent=2))